<a href="https://colab.research.google.com/github/AlperYildirim1/crt-fourier-transformer-addition/blob/main/T2_T5_Rotates_T10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================================
# TEST 13 — DOES EXCLUSIVE T2/T5 CAUSALLY ROTATE DOWNSTREAM T10?
# Pythia-6.9B, two-digit addition, standalone Colab script
#
# Core question:
#   If we change only the fitted T2/T5 answer-position code while leaving the
#   fitted T10 coordinate untouched at the intervention layer, does downstream
#   computation rotate T10 toward the CRT-implied target digit?
#
# Modes:
#   1. Single-layer sweep:
#        intervene once at L, for every L in SINGLE_LAYERS.
#   2. Persistent steering:
#        re-apply the T2/T5 target at every layer 18..31.
#
# Conditions in each mode:
#   keep:
#        remove/reinsert A's own exclusive T2/T5 content (surgical control).
#   target:
#        keep A's T2 and T5 radii, insert B's T2 sign and T5 phase.
#
# Critical construction:
#   Q25_exclusive is made orthogonal to raw Q10 at every layer. Therefore the
#   intervention cannot directly write T10 at that same layer. Any later T10
#   movement must be produced downstream by the model.
#
# Main trajectory metric:
#   q10_score = cosine(run_Q10, clean_B_Q10) - cosine(run_Q10, clean_A_Q10)
#
# Interpretation:
#   target-minus-keep q10_score > 0 downstream:
#       evidence that T2/T5 causally rotates/reconstructs T10.
#   output follows B while downstream Q10 does not move:
#       evidence for a T2/T5 output route parallel to T10.
#
# Recommended first run:
#   SMOKE_TEST = True
#
# Colab install:
# !pip install -q transformers accelerate scikit-learn pandas tqdm
# =============================================================================

import gc
import json
import os
import random
from dataclasses import dataclass
from typing import Dict, List, Sequence

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import Ridge
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer


# =============================================================================
# CONFIG
# =============================================================================

MODEL_NAME = "EleutherAI/pythia-6.9b"
OUT_DIR = "/content/pythia_test13_q25_to_q10_propagation"

SEED = 42
RIDGE_ALPHA = 1.0
BATCH_SIZE = 12

SMOKE_TEST = False
FIT_N = 600 if SMOKE_TEST else 4000
EVAL_N = 200 if SMOKE_TEST else 1500

START_LAYER = 18
END_LAYER = 31
SINGLE_LAYERS = list(range(START_LAYER, END_LAYER + 1))
PERSISTENT_LAYERS = list(range(START_LAYER, END_LAYER + 1))
MEASUREMENT_LAYERS = list(range(START_LAYER, END_LAYER + 1))

RUN_KEEP_CONTROLS = True
RUN_SINGLE_LAYER_SWEEP = True
RUN_PERSISTENT = True

A_MAX = 99
B_MAX = 99


# =============================================================================
# UTILITIES
# =============================================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_prompt(a: int, b: int) -> str:
    return f"Output ONLY a number. {a}+{b}="


def answer_token_id(tokenizer, n: int):
    ids = tokenizer(str(int(n)), add_special_tokens=False)["input_ids"]
    return int(ids[0]) if len(ids) == 1 else None


def pred_token_to_int(tokenizer, token_id: int):
    text = tokenizer.decode([int(token_id)]).strip()
    try:
        return int(text)
    except Exception:
        return None


def get_last_nonpad_positions(attention_mask: torch.Tensor) -> torch.Tensor:
    positions = torch.arange(
        attention_mask.shape[1],
        device=attention_mask.device,
    )
    return (attention_mask * positions.unsqueeze(0)).max(dim=1).values.long()


def get_blocks(model):
    if hasattr(model, "gpt_neox") and hasattr(model.gpt_neox, "layers"):
        return model.gpt_neox.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    raise RuntimeError("Could not find transformer blocks.")


def orthonormalize_columns(matrix: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    U, singular_values, _ = np.linalg.svd(matrix, full_matrices=False)
    threshold = eps * max(float(singular_values[0]), eps)
    rank = int(np.sum(singular_values > threshold))
    if rank == 0:
        raise RuntimeError("Matrix has zero numerical rank.")
    return U[:, :rank]


def safe_unit(coordinates: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    return coordinates / (coordinates.norm(dim=1, keepdim=True) + eps)


def cosine_rows(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    return torch.nn.functional.cosine_similarity(a, b, dim=1, eps=1e-8)


def circular_abs_error(angle_a: torch.Tensor, angle_b: torch.Tensor) -> torch.Tensor:
    delta = torch.atan2(torch.sin(angle_a - angle_b), torch.cos(angle_a - angle_b))
    return delta.abs()


# =============================================================================
# MODEL
# =============================================================================

set_seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
).eval()
model.config.pad_token_id = tokenizer.pad_token_id

blocks = get_blocks(model)
N_LAYERS = len(blocks)
SINGLE_LAYERS = [layer for layer in SINGLE_LAYERS if 0 <= layer < N_LAYERS]
PERSISTENT_LAYERS = [
    layer
    for layer in PERSISTENT_LAYERS
    if 0 <= layer < N_LAYERS
]
MEASUREMENT_LAYERS = [
    layer
    for layer in MEASUREMENT_LAYERS
    if 0 <= layer < N_LAYERS
]

if not SINGLE_LAYERS or not PERSISTENT_LAYERS:
    raise RuntimeError("No valid intervention layers.")

print("model layers:", N_LAYERS)
print("single-layer sweep:", SINGLE_LAYERS)
print("persistent layers:", PERSISTENT_LAYERS)


# =============================================================================
# DATASET AND DISJOINT FIT/EVALUATION SPLIT
# =============================================================================

def make_examples() -> List[dict]:
    rows = []
    for a in range(A_MAX + 1):
        for b in range(B_MAX + 1):
            total = a + b
            target_token_id = answer_token_id(tokenizer, total)
            if target_token_id is None:
                continue
            rows.append({
                "a": a,
                "b": b,
                "sum": total,
                "units": total % 10,
                "target_token_id": target_token_id,
                "prompt": make_prompt(a, b),
            })
    return rows


@torch.no_grad()
def baseline_filter(rows: Sequence[dict]) -> List[dict]:
    kept = []
    for start in tqdm(range(0, len(rows), BATCH_SIZE), desc="baseline filter"):
        batch = rows[start:start + BATCH_SIZE]
        encoded = tokenizer(
            [row["prompt"] for row in batch],
            return_tensors="pt",
            padding=True,
            add_special_tokens=False,
        ).to(model.device)
        positions = get_last_nonpad_positions(encoded["attention_mask"])
        batch_indices = torch.arange(len(batch), device=model.device)
        logits = model(**encoded, use_cache=False).logits[batch_indices, positions]
        prediction_ids = logits.argmax(dim=-1).detach().cpu().tolist()
        for row, prediction_id in zip(batch, prediction_ids):
            if int(prediction_id) == int(row["target_token_id"]):
                kept.append(row)
    return kept


all_examples = make_examples()
print("candidate examples:", len(all_examples))
baseline_correct = baseline_filter(all_examples)
print("baseline correct:", len(baseline_correct), "/", len(all_examples))

split_rng = np.random.default_rng(SEED)
split_rng.shuffle(baseline_correct)

fit_count = min(FIT_N, len(baseline_correct))
fit_rows = baseline_correct[:fit_count]
eval_pool = baseline_correct[fit_count:]

if len(fit_rows) < 100 or len(eval_pool) < 100:
    raise RuntimeError("Not enough disjoint baseline-correct examples.")

print("fit rows:", len(fit_rows))
print("evaluation pool:", len(eval_pool))


# =============================================================================
# COLLECT CLEAN ANSWER-POSITION RESIDUALS FOR FITTING
# =============================================================================

@torch.no_grad()
def collect_equal_residuals(rows: Sequence[dict], layers: Sequence[int]):
    store = {layer: [] for layer in layers}
    sums = []
    buffer: Dict[int, torch.Tensor] = {}

    def make_capture_hook(layer: int):
        def hook(module, inputs, output):
            hidden = output[0] if isinstance(output, tuple) else output
            buffer[layer] = hidden.detach()
        return hook

    handles = [
        blocks[layer].register_forward_hook(make_capture_hook(layer))
        for layer in layers
    ]
    try:
        for start in tqdm(
            range(0, len(rows), BATCH_SIZE),
            desc='collect "=" residuals',
        ):
            batch = rows[start:start + BATCH_SIZE]
            encoded = tokenizer(
                [row["prompt"] for row in batch],
                return_tensors="pt",
                padding=True,
                add_special_tokens=False,
            ).to(model.device)
            positions = get_last_nonpad_positions(encoded["attention_mask"])
            buffer.clear()
            model(**encoded, use_cache=False)

            for layer in layers:
                hidden = buffer[layer]
                local_positions = positions.to(hidden.device)
                local_indices = torch.arange(len(batch), device=hidden.device)
                vectors = hidden[local_indices, local_positions]
                store[layer].append(vectors.float().cpu().numpy())

            sums.extend(int(row["sum"]) for row in batch)
    finally:
        for handle in handles:
            handle.remove()

    return (
        {
            layer: np.concatenate(store[layer], axis=0)
            for layer in layers
        },
        np.asarray(sums, dtype=np.float64),
    )


X_fit, fit_sums = collect_equal_residuals(fit_rows, MEASUREMENT_LAYERS)


# =============================================================================
# FIT RAW Q2/Q5/Q10 AND Q25 EXCLUSIVE OF Q10
# =============================================================================

@dataclass
class GearBases:
    q2_exclusive: torch.Tensor
    q5_exclusive: torch.Tensor
    q25_exclusive: torch.Tensor
    q10_raw: torch.Tensor
    q10_digit_centroids: torch.Tensor
    raw_overlap_cosine_max: float
    exclusive_overlap_cosine_max: float
    q25_retained_frobenius: float


def fit_gear_bases(X: np.ndarray, sums: np.ndarray) -> GearBases:
    y2 = np.cos(np.pi * sums).reshape(-1, 1)
    theta5 = 2.0 * np.pi * sums / 5.0
    y5 = np.column_stack([np.cos(theta5), np.sin(theta5)])
    theta10 = 2.0 * np.pi * sums / 10.0
    y10 = np.column_stack([np.cos(theta10), np.sin(theta10)])

    w2 = Ridge(alpha=RIDGE_ALPHA, fit_intercept=True).fit(y2, X).coef_
    w5 = Ridge(alpha=RIDGE_ALPHA, fit_intercept=True).fit(y5, X).coef_
    w10 = Ridge(alpha=RIDGE_ALPHA, fit_intercept=True).fit(y10, X).coef_

    q2_raw = orthonormalize_columns(w2)[:, :1]
    w5_no_q2 = w5 - q2_raw @ (q2_raw.T @ w5)
    q5_raw = orthonormalize_columns(w5_no_q2)
    if q5_raw.shape[1] < 2:
        raise RuntimeError("Raw T5 rank < 2.")
    q5_raw = q5_raw[:, :2]
    q25_raw, _ = np.linalg.qr(np.concatenate([q2_raw, q5_raw], axis=1))
    q25_raw = q25_raw[:, :3]

    q10_raw = orthonormalize_columns(w10)
    if q10_raw.shape[1] < 2:
        raise RuntimeError("Raw T10 rank < 2.")
    q10_raw = q10_raw[:, :2]

    # Remove every raw-Q10 component from the T2 and T5 fitted directions.
    w2_exclusive = w2 - q10_raw @ (q10_raw.T @ w2)
    q2_exclusive = orthonormalize_columns(w2_exclusive)[:, :1]

    w5_exclusive = w5 - q10_raw @ (q10_raw.T @ w5)
    w5_exclusive = (
        w5_exclusive
        - q2_exclusive @ (q2_exclusive.T @ w5_exclusive)
    )
    q5_exclusive = orthonormalize_columns(w5_exclusive)
    if q5_exclusive.shape[1] < 2:
        raise RuntimeError("T5 exclusive of T10 rank < 2.")
    q5_exclusive = q5_exclusive[:, :2]
    q25_exclusive, _ = np.linalg.qr(
        np.concatenate([q2_exclusive, q5_exclusive], axis=1)
    )
    q25_exclusive = q25_exclusive[:, :3]

    raw_overlap = np.linalg.svd(q25_raw.T @ q10_raw, compute_uv=False)
    exclusive_overlap = np.linalg.svd(
        q25_exclusive.T @ q10_raw,
        compute_uv=False,
    )

    w25 = np.concatenate([w2, w5], axis=1)
    w25_exclusive = w25 - q10_raw @ (q10_raw.T @ w25)
    retained = float(
        np.linalg.norm(w25_exclusive, ord="fro")
        / (np.linalg.norm(w25, ord="fro") + 1e-12)
    )

    # Empirical clean digit centroids in the fitted raw-Q10 coordinates.
    q10_coordinates = X @ q10_raw
    centroids = []
    for digit in range(10):
        mask = (sums.astype(int) % 10) == digit
        if not np.any(mask):
            raise RuntimeError(f"No fit examples for digit {digit}.")
        centroids.append(q10_coordinates[mask].mean(axis=0))
    centroids = np.stack(centroids, axis=0)

    return GearBases(
        q2_exclusive=torch.tensor(q2_exclusive, dtype=torch.float32),
        q5_exclusive=torch.tensor(q5_exclusive, dtype=torch.float32),
        q25_exclusive=torch.tensor(q25_exclusive, dtype=torch.float32),
        q10_raw=torch.tensor(q10_raw, dtype=torch.float32),
        q10_digit_centroids=torch.tensor(centroids, dtype=torch.float32),
        raw_overlap_cosine_max=float(raw_overlap.max()),
        exclusive_overlap_cosine_max=float(exclusive_overlap.max()),
        q25_retained_frobenius=retained,
    )


bases_by_layer = {
    layer: fit_gear_bases(X_fit[layer], fit_sums)
    for layer in MEASUREMENT_LAYERS
}

basis_audit_df = pd.DataFrame([
    {
        "layer": layer,
        "raw_overlap_cosine_max": bases.raw_overlap_cosine_max,
        "exclusive_overlap_cosine_max": bases.exclusive_overlap_cosine_max,
        "q25_retained_frobenius": bases.q25_retained_frobenius,
    }
    for layer, bases in bases_by_layer.items()
])
basis_audit_path = os.path.join(OUT_DIR, "test13_basis_audit.csv")
basis_audit_df.to_csv(basis_audit_path, index=False)

del X_fit
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


# =============================================================================
# BUILD BALANCED COUNTERFACTUAL A/B PAIRS
# =============================================================================

def build_pairs(pool: Sequence[dict], n: int, seed: int) -> List[dict]:
    local_rng = np.random.default_rng(seed)
    pool = list(pool)
    by_units = {
        units: [row for row in pool if row["units"] == units]
        for units in range(10)
    }

    n = min(n, len(pool))
    a_indices = local_rng.permutation(len(pool))[:n]
    deltas = np.resize(np.arange(1, 10, dtype=int), n)
    local_rng.shuffle(deltas)

    pairs = []
    for pair_id, (index, delta) in enumerate(zip(a_indices, deltas)):
        A = pool[int(index)]
        target_units = int((A["units"] + int(delta)) % 10)
        candidates = by_units[target_units]
        if not candidates:
            raise RuntimeError(f"No donor candidates for digit {target_units}.")
        B = candidates[int(local_rng.integers(len(candidates)))]
        pairs.append({
            "pair_id": pair_id,
            "A": A,
            "B": B,
            "delta_mod10": int(delta),
            "target_mod2": B["sum"] % 2,
            "target_mod5": B["sum"] % 5,
            "target_mod10": B["sum"] % 10,
        })
    return pairs


pairs = build_pairs(eval_pool, min(EVAL_N, len(eval_pool)), SEED + 1)
print("evaluation pairs:", len(pairs))


# =============================================================================
# PRECOMPUTE CLEAN A/B Q25 AND Q10 COORDINATES
# =============================================================================

@dataclass
class CleanCoordinates:
    a2: torch.Tensor
    b2: torch.Tensor
    a5: torch.Tensor
    b5: torch.Tensor
    a10: torch.Tensor
    b10: torch.Tensor


@torch.no_grad()
def precompute_clean_coordinates(
    pairs: Sequence[dict],
    layers: Sequence[int],
) -> Dict[int, CleanCoordinates]:
    keys = ["a2", "b2", "a5", "b5", "a10", "b10"]
    stores = {
        layer: {key: [] for key in keys}
        for layer in layers
    }
    captured: Dict[int, torch.Tensor] = {}

    def make_capture_hook(layer: int):
        def hook(module, inputs, output):
            hidden = output[0] if isinstance(output, tuple) else output
            captured[layer] = hidden.detach()
        return hook

    handles = [
        blocks[layer].register_forward_hook(make_capture_hook(layer))
        for layer in layers
    ]
    try:
        for start in tqdm(
            range(0, len(pairs), BATCH_SIZE),
            desc="precompute clean A/B coordinates",
        ):
            batch = pairs[start:start + BATCH_SIZE]
            batch_size = len(batch)
            rows = (
                [pair["A"] for pair in batch]
                + [pair["B"] for pair in batch]
            )
            encoded = tokenizer(
                [row["prompt"] for row in rows],
                return_tensors="pt",
                padding=True,
                add_special_tokens=False,
            ).to(model.device)
            positions = get_last_nonpad_positions(encoded["attention_mask"])
            captured.clear()
            model(**encoded, use_cache=False)

            for layer in layers:
                hidden = captured[layer]
                local_positions = positions.to(hidden.device)
                local_indices = torch.arange(2 * batch_size, device=hidden.device)
                vectors = hidden[local_indices, local_positions].float()
                hA = vectors[:batch_size]
                hB = vectors[batch_size:]
                bases = bases_by_layer[layer]
                q2 = bases.q2_exclusive.to(hidden.device)
                q5 = bases.q5_exclusive.to(hidden.device)
                q10 = bases.q10_raw.to(hidden.device)

                stores[layer]["a2"].append((hA @ q2).cpu())
                stores[layer]["b2"].append((hB @ q2).cpu())
                stores[layer]["a5"].append((hA @ q5).cpu())
                stores[layer]["b5"].append((hB @ q5).cpu())
                stores[layer]["a10"].append((hA @ q10).cpu())
                stores[layer]["b10"].append((hB @ q10).cpu())
    finally:
        for handle in handles:
            handle.remove()

    return {
        layer: CleanCoordinates(
            **{
                key: torch.cat(stores[layer][key], dim=0)
                for key in keys
            }
        )
        for layer in layers
    }


clean_coordinates_by_layer = precompute_clean_coordinates(
    pairs,
    MEASUREMENT_LAYERS,
)


# =============================================================================
# HELD-OUT Q10 MEASUREMENT CALIBRATION
# =============================================================================

calibration_rows = []
for layer in MEASUREMENT_LAYERS:
    coordinates = clean_coordinates_by_layer[layer].a10
    centroids = bases_by_layer[layer].q10_digit_centroids
    distances = torch.sum(
        (coordinates[:, None, :] - centroids[None, :, :]) ** 2,
        dim=2,
    )
    predicted_digits = distances.argmin(dim=1).numpy()
    true_digits = np.asarray([pair["A"]["units"] for pair in pairs], dtype=int)
    calibration_rows.append({
        "layer": layer,
        "n": len(pairs),
        "q10_digit_accuracy": float(np.mean(predicted_digits == true_digits)),
        "chance": 0.1,
    })

q10_calibration_df = pd.DataFrame(calibration_rows)
q10_calibration_path = os.path.join(OUT_DIR, "test13_q10_calibration.csv")
q10_calibration_df.to_csv(q10_calibration_path, index=False)


# =============================================================================
# INTERVENTION TARGETS AND HOOKS
# =============================================================================

def build_q25_contribution(
    clean: CleanCoordinates,
    bases: GearBases,
    row_slice: slice,
    condition: str,
) -> torch.Tensor:
    a2 = clean.a2[row_slice]
    a5 = clean.a5[row_slice]

    if condition == "keep":
        new2 = a2
        new5 = a5
    elif condition == "target":
        b2 = clean.b2[row_slice]
        b5 = clean.b5[row_slice]
        new2 = a2.abs() * torch.sign(b2)
        new5 = a5.norm(dim=1, keepdim=True) * safe_unit(b5)
    else:
        raise ValueError(condition)

    return (
        new2 @ bases.q2_exclusive.T
        + new5 @ bases.q5_exclusive.T
    )


def make_measure_or_intervene_hook(
    layer: int,
    positions: torch.Tensor,
    batch_indices: torch.Tensor,
    intervene: bool,
    target_contribution: torch.Tensor,
    q10_record: dict,
    direct_audit: dict,
):
    def hook(module, inputs, output):
        if isinstance(output, tuple):
            hidden = output[0].clone() if intervene else output[0]
            rest = tuple(output[1:])
        else:
            hidden = output.clone() if intervene else output
            rest = None

        local_positions = positions.to(hidden.device)
        local_indices = batch_indices.to(hidden.device)
        vector = hidden[local_indices, local_positions]
        bases = bases_by_layer[layer]
        q10 = bases.q10_raw.to(vector.device, dtype=vector.dtype)

        before_q10 = vector @ q10

        if intervene:
            q25 = bases.q25_exclusive.to(vector.device, dtype=vector.dtype)
            target = target_contribution.to(vector.device, dtype=vector.dtype)
            edited = vector - (vector @ q25) @ q25.T + target
            hidden[local_indices, local_positions] = edited
            after_q10 = edited @ q10

            numerator = (after_q10.float() - before_q10.float()).norm(dim=1)
            denominator = before_q10.float().norm(dim=1).clamp_min(1e-8)
            direct_audit[layer]["count"] += int(vector.shape[0])
            direct_audit[layer]["sum_relative_q10_change"] += float(
                (numerator / denominator).sum().item()
            )
        else:
            after_q10 = before_q10

        q10_record[layer] = after_q10.detach().float().cpu()
        return hidden if rest is None else (hidden,) + rest

    return hook


# =============================================================================
# RUN ONE MODE/CONDITION
# =============================================================================

@torch.no_grad()
def evaluate_mode_condition(
    pairs: Sequence[dict],
    mode: str,
    condition: str,
    single_layer: int = None,
):
    if mode == "single":
        intervention_layers = [int(single_layer)]
        measurement_layers = [
            layer
            for layer in MEASUREMENT_LAYERS
            if layer >= int(single_layer)
        ]
        intervention_id = int(single_layer)
    elif mode == "persistent":
        intervention_layers = list(PERSISTENT_LAYERS)
        measurement_layers = list(MEASUREMENT_LAYERS)
        intervention_id = min(PERSISTENT_LAYERS)
    else:
        raise ValueError(mode)

    counts = {
        "n": 0,
        "parseable": 0,
        "exact_A": 0,
        "stay_A_mod10": 0,
        "follow_target_mod10": 0,
    }
    output_records = []
    trajectory_records = []
    direct_audit = {
        layer: {"count": 0, "sum_relative_q10_change": 0.0}
        for layer in intervention_layers
    }

    for start in tqdm(
        range(0, len(pairs), BATCH_SIZE),
        desc=f"{mode} {intervention_id} {condition}",
        leave=False,
    ):
        batch = pairs[start:start + BATCH_SIZE]
        row_slice = slice(start, start + len(batch))
        A_rows = [pair["A"] for pair in batch]
        encoded = tokenizer(
            [row["prompt"] for row in A_rows],
            return_tensors="pt",
            padding=True,
            add_special_tokens=False,
        ).to(model.device)
        positions = get_last_nonpad_positions(encoded["attention_mask"])
        batch_indices = torch.arange(len(batch), device=model.device)

        target_by_layer = {
            layer: build_q25_contribution(
                clean_coordinates_by_layer[layer],
                bases_by_layer[layer],
                row_slice,
                condition,
            )
            for layer in intervention_layers
        }

        q10_record = {}
        handles = []
        try:
            for layer in measurement_layers:
                intervene = layer in intervention_layers
                handles.append(
                    blocks[layer].register_forward_hook(
                        make_measure_or_intervene_hook(
                            layer=layer,
                            positions=positions,
                            batch_indices=batch_indices,
                            intervene=intervene,
                            target_contribution=(
                                target_by_layer[layer]
                                if intervene
                                else None
                            ),
                            q10_record=q10_record,
                            direct_audit=direct_audit,
                        )
                    )
                )

            logits = model(**encoded, use_cache=False).logits[
                batch_indices,
                positions,
            ]
            prediction_ids = logits.argmax(dim=-1).detach().cpu().tolist()
        finally:
            for handle in handles:
                handle.remove()

        for local_index, (pair, prediction_id) in enumerate(
            zip(batch, prediction_ids)
        ):
            A = pair["A"]
            prediction = pred_token_to_int(tokenizer, prediction_id)
            parseable = int(prediction is not None)
            exact_A = int(int(prediction_id) == int(A["target_token_id"]))
            stay_A = int(parseable and prediction % 10 == A["units"])
            follow_target = int(
                parseable and prediction % 10 == pair["target_mod10"]
            )

            counts["n"] += 1
            counts["parseable"] += parseable
            counts["exact_A"] += exact_A
            counts["stay_A_mod10"] += stay_A
            counts["follow_target_mod10"] += follow_target

            output_records.append({
                "pair_id": pair["pair_id"],
                "mode": mode,
                "condition": condition,
                "intervention_layer": intervention_id,
                "a_A": A["a"],
                "b_A": A["b"],
                "sum_A": A["sum"],
                "units_A": A["units"],
                "sum_B": pair["B"]["sum"],
                "target_mod10": pair["target_mod10"],
                "delta_mod10": pair["delta_mod10"],
                "pred": prediction,
                "parseable": parseable,
                "exact_A": exact_A,
                "stay_A_mod10": stay_A,
                "follow_target_mod10": follow_target,
            })

            global_index = start + local_index
            for measurement_layer in measurement_layers:
                run_q10 = q10_record[measurement_layer][local_index:local_index + 1]
                clean = clean_coordinates_by_layer[measurement_layer]
                a_q10 = clean.a10[global_index:global_index + 1]
                b_q10 = clean.b10[global_index:global_index + 1]

                stay_cosine = float(cosine_rows(run_q10, a_q10).item())
                target_cosine = float(cosine_rows(run_q10, b_q10).item())
                q10_score = target_cosine - stay_cosine

                run_angle = torch.atan2(run_q10[:, 1], run_q10[:, 0])
                a_angle = torch.atan2(a_q10[:, 1], a_q10[:, 0])
                b_angle = torch.atan2(b_q10[:, 1], b_q10[:, 0])

                trajectory_records.append({
                    "pair_id": pair["pair_id"],
                    "mode": mode,
                    "condition": condition,
                    "intervention_layer": intervention_id,
                    "measurement_layer": measurement_layer,
                    "layers_after_intervention": (
                        measurement_layer - intervention_id
                    ),
                    "delta_mod10": pair["delta_mod10"],
                    "stay_q10_cosine": stay_cosine,
                    "target_q10_cosine": target_cosine,
                    "q10_target_minus_stay": q10_score,
                    "closer_to_target_q10": int(target_cosine > stay_cosine),
                    "stay_q10_angle_error": float(
                        circular_abs_error(run_angle, a_angle).item()
                    ),
                    "target_q10_angle_error": float(
                        circular_abs_error(run_angle, b_angle).item()
                    ),
                })

    n = max(counts["n"], 1)
    summary = {
        "mode": mode,
        "condition": condition,
        "intervention_layer": intervention_id,
        "last_measurement_layer": max(measurement_layers),
        "n": counts["n"],
        "parseable": counts["parseable"] / n,
        "exact_A": counts["exact_A"] / n,
        "stay_A_mod10": counts["stay_A_mod10"] / n,
        "follow_target_mod10": counts["follow_target_mod10"] / n,
        "net_target_vs_stay10": (
            counts["follow_target_mod10"] - counts["stay_A_mod10"]
        ) / n,
    }

    audit_rows = []
    for layer, audit in direct_audit.items():
        count = max(audit["count"], 1)
        audit_rows.append({
            "mode": mode,
            "condition": condition,
            "intervention_layer": intervention_id,
            "edited_layer": layer,
            "n_vectors": audit["count"],
            "mean_immediate_relative_q10_change": (
                audit["sum_relative_q10_change"] / count
            ),
        })

    return summary, output_records, trajectory_records, audit_rows


# =============================================================================
# RUN SINGLE-LAYER SWEEP AND CONTINUAL/PERSISTENT CONDITIONS
# =============================================================================

conditions = ["target"]
if RUN_KEEP_CONTROLS:
    conditions = ["keep", "target"]

all_summaries = []
all_output_records = []
all_trajectory_records = []
all_direct_audits = []

if RUN_SINGLE_LAYER_SWEEP:
    for layer in SINGLE_LAYERS:
        for condition in conditions:
            summary, outputs, trajectories, audits = evaluate_mode_condition(
                pairs=pairs,
                mode="single",
                condition=condition,
                single_layer=layer,
            )
            all_summaries.append(summary)
            all_output_records.extend(outputs)
            all_trajectory_records.extend(trajectories)
            all_direct_audits.extend(audits)
            print(summary)

            pd.DataFrame(all_summaries).to_csv(
                os.path.join(OUT_DIR, "test13_summary_partial.csv"),
                index=False,
            )
            pd.DataFrame(all_trajectory_records).to_csv(
                os.path.join(OUT_DIR, "test13_trajectory_partial.csv"),
                index=False,
            )

if RUN_PERSISTENT:
    for condition in conditions:
        summary, outputs, trajectories, audits = evaluate_mode_condition(
            pairs=pairs,
            mode="persistent",
            condition=condition,
        )
        all_summaries.append(summary)
        all_output_records.extend(outputs)
        all_trajectory_records.extend(trajectories)
        all_direct_audits.extend(audits)
        print(summary)

summary_df = pd.DataFrame(all_summaries)
output_records_df = pd.DataFrame(all_output_records)
trajectory_df = pd.DataFrame(all_trajectory_records)
direct_audit_df = pd.DataFrame(all_direct_audits)

summary_path = os.path.join(OUT_DIR, "test13_output_summary.csv")
output_records_path = os.path.join(OUT_DIR, "test13_output_predictions.csv")
trajectory_path = os.path.join(OUT_DIR, "test13_q10_trajectory_records.csv")
direct_audit_path = os.path.join(OUT_DIR, "test13_immediate_q10_audit.csv")

summary_df.to_csv(summary_path, index=False)
output_records_df.to_csv(output_records_path, index=False)
trajectory_df.to_csv(trajectory_path, index=False)
direct_audit_df.to_csv(direct_audit_path, index=False)


# =============================================================================
# PAIRED TARGET-vs-KEEP PROPAGATION AND OUTPUT CONTRASTS
# =============================================================================

def summarize_paired_trajectory(trajectory: pd.DataFrame) -> pd.DataFrame:
    if "keep" not in set(trajectory["condition"]):
        return pd.DataFrame()

    pivot = trajectory.pivot_table(
        index=[
            "pair_id",
            "mode",
            "intervention_layer",
            "measurement_layer",
            "layers_after_intervention",
        ],
        columns="condition",
        values="q10_target_minus_stay",
    ).reset_index()
    pivot["target_minus_keep_q10_score"] = pivot["target"] - pivot["keep"]

    rows = []
    group_columns = [
        "mode",
        "intervention_layer",
        "measurement_layer",
        "layers_after_intervention",
    ]
    for group_key, group in pivot.groupby(group_columns):
        values = group["target_minus_keep_q10_score"].dropna().to_numpy()
        standard_error = float(values.std(ddof=1) / np.sqrt(len(values)))
        rows.append({
            **dict(zip(group_columns, group_key)),
            "n": int(len(values)),
            "mean_target_minus_keep_q10_score": float(values.mean()),
            "standard_error": standard_error,
            "ci95_low": float(values.mean() - 1.96 * standard_error),
            "ci95_high": float(values.mean() + 1.96 * standard_error),
        })
    return pd.DataFrame(rows)


def summarize_paired_output(outputs: pd.DataFrame) -> pd.DataFrame:
    if "keep" not in set(outputs["condition"]):
        return pd.DataFrame()

    rows = []
    for metric in ["follow_target_mod10", "stay_A_mod10", "exact_A"]:
        pivot = outputs.pivot_table(
            index=["pair_id", "mode", "intervention_layer"],
            columns="condition",
            values=metric,
        ).reset_index()
        pivot["difference"] = pivot["target"] - pivot["keep"]

        for group_key, group in pivot.groupby(["mode", "intervention_layer"]):
            values = group["difference"].dropna().to_numpy(dtype=float)
            standard_error = float(values.std(ddof=1) / np.sqrt(len(values)))
            rows.append({
                "mode": group_key[0],
                "intervention_layer": group_key[1],
                "metric": metric,
                "n": int(len(values)),
                "mean_target_minus_keep": float(values.mean()),
                "ci95_low": float(values.mean() - 1.96 * standard_error),
                "ci95_high": float(values.mean() + 1.96 * standard_error),
            })
    return pd.DataFrame(rows)


paired_trajectory_df = summarize_paired_trajectory(trajectory_df)
paired_output_df = summarize_paired_output(output_records_df)

paired_trajectory_path = os.path.join(
    OUT_DIR,
    "test13_paired_q10_propagation.csv",
)
paired_output_path = os.path.join(OUT_DIR, "test13_paired_output_contrasts.csv")
paired_trajectory_df.to_csv(paired_trajectory_path, index=False)
paired_output_df.to_csv(paired_output_path, index=False)

trajectory_by_delta_df = (
    trajectory_df
    .groupby(
        [
            "mode",
            "condition",
            "intervention_layer",
            "measurement_layer",
            "delta_mod10",
        ],
        as_index=False,
    )
    .agg(
        n=("pair_id", "count"),
        mean_q10_target_minus_stay=("q10_target_minus_stay", "mean"),
        closer_to_target_rate=("closer_to_target_q10", "mean"),
        mean_target_angle_error=("target_q10_angle_error", "mean"),
    )
)
trajectory_by_delta_path = os.path.join(
    OUT_DIR,
    "test13_q10_trajectory_by_delta.csv",
)
trajectory_by_delta_df.to_csv(trajectory_by_delta_path, index=False)


# =============================================================================
# CONFIG AND COMPACT FINAL PRINT
# =============================================================================

config = {
    "model_name": MODEL_NAME,
    "seed": SEED,
    "ridge_alpha": RIDGE_ALPHA,
    "fit_n": len(fit_rows),
    "eval_n": len(pairs),
    "fit_eval_disjoint": True,
    "single_layers": SINGLE_LAYERS,
    "persistent_layers": PERSISTENT_LAYERS,
    "measurement_layers": MEASUREMENT_LAYERS,
    "run_keep_controls": RUN_KEEP_CONTROLS,
    "smoke_test": SMOKE_TEST,
}
config_path = os.path.join(OUT_DIR, "test13_config.json")
with open(config_path, "w") as file:
    json.dump(config, file, indent=2)

print("\n" + "=" * 100)
print("OUTPUT SUMMARY")
print("=" * 100)
print(summary_df.to_string(index=False))

if not paired_output_df.empty:
    print("\nPAIRED OUTPUT CONTRASTS")
    print(paired_output_df.to_string(index=False))

if not paired_trajectory_df.empty:
    print("\nFINAL-LAYER Q10 PROPAGATION")
    final_layer_rows = paired_trajectory_df[
        paired_trajectory_df["measurement_layer"] == max(MEASUREMENT_LAYERS)
    ]
    print(final_layer_rows.to_string(index=False))

    print("\nPERSISTENT Q10 TRAJECTORY")
    persistent_rows = paired_trajectory_df[
        paired_trajectory_df["mode"] == "persistent"
    ]
    print(persistent_rows.to_string(index=False))

print("\nQ10 MEASUREMENT CALIBRATION")
print(q10_calibration_df.to_string(index=False))

print("\nIMMEDIATE-CHANGE AUDIT")
audit_summary = (
    direct_audit_df
    .groupby(["mode", "condition"], as_index=False)
    .agg(
        max_immediate_relative_q10_change=(
            "mean_immediate_relative_q10_change",
            "max",
        )
    )
)
print(audit_summary.to_string(index=False))

print("\nSaved under:", OUT_DIR)
print("output summary:", summary_path)
print("paired output:", paired_output_path)
print("paired Q10 propagation:", paired_trajectory_path)
print("trajectory records:", trajectory_path)
print("trajectory by delta:", trajectory_by_delta_path)
print("Q10 calibration:", q10_calibration_path)
print("immediate Q10 audit:", direct_audit_path)
print("basis audit:", basis_audit_path)
print("config:", config_path)